In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("fraud-detection") \
    .config("spark.jars.packages", 
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark version:", spark.version)
print("✅ Spark session created successfully")

Spark version: 3.5.0
✅ Spark session created successfully


In [2]:
from delta.tables import DeltaTable
import pandas as pd

# Write a small test Delta table to the bronze bucket
test_data = spark.createDataFrame([
    (1, "test_transaction", 100.0, 0),
    (2, "test_transaction", 250.0, 1),
    (3, "test_transaction", 50.0, 0),
], ["id", "type", "amount", "is_fraud"])

test_data.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://bronze/test/")

print("✅ Write to MinIO successful")

# Read it back
df_read = spark.read.format("delta").load("s3a://bronze/test/")
df_read.show()
print("✅ Read from MinIO successful")

✅ Write to MinIO successful
+---+----------------+------+--------+
| id|            type|amount|is_fraud|
+---+----------------+------+--------+
|  3|test_transaction|  50.0|       0|
|  1|test_transaction| 100.0|       0|
|  2|test_transaction| 250.0|       1|
+---+----------------+------+--------+

✅ Read from MinIO successful


In [5]:
from pyspark.sql.types import *

# Bronze schema - raw data + audit columns
bronze_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("step", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", IntegerType(), True),
    # Audit columns
    StructField("ingestion_ts", TimestampType(), True),
    StructField("source", StringType(), True),
])

print("✅ Bronze schema defined")
print(f"   Fields: {len(bronze_schema.fields)}")
for f in bronze_schema.fields:
    print(f"   - {f.name}: {f.dataType}")

✅ Bronze schema defined
   Fields: 14
   - transaction_id: StringType()
   - step: IntegerType()
   - type: StringType()
   - amount: DoubleType()
   - nameOrig: StringType()
   - oldbalanceOrg: DoubleType()
   - newbalanceOrig: DoubleType()
   - nameDest: StringType()
   - oldbalanceDest: DoubleType()
   - newbalanceDest: DoubleType()
   - isFraud: IntegerType()
   - isFlaggedFraud: IntegerType()
   - ingestion_ts: TimestampType()
   - source: StringType()


In [6]:
# Silver schema - cleaned + typed
silver_schema = StructType([
    StructField("transaction_id", StringType(), False),  # not nullable after cleaning
    StructField("step", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("nameOrig", StringType(), False),
    StructField("oldbalanceOrg", DoubleType(), False),
    StructField("newbalanceOrig", DoubleType(), False),
    StructField("nameDest", StringType(), False),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), False),
    StructField("isFlaggedFraud", IntegerType(), False),
    # Derived features
    StructField("balance_diff_orig", DoubleType(), True),
    StructField("balance_diff_dest", DoubleType(), True),
    StructField("transaction_date", DateType(), False),
    # Audit
    StructField("ingestion_ts", TimestampType(), False),
    StructField("source", StringType(), False),
    StructField("processed_ts", TimestampType(), False),
])

# Gold schema - business aggregates
gold_schema = StructType([
    StructField("transaction_date", DateType(), False),
    StructField("type", StringType(), False),
    StructField("total_transactions", LongType(), False),
    StructField("total_amount", DoubleType(), False),
    StructField("fraud_count", LongType(), False),
    StructField("fraud_rate", DoubleType(), False),
    StructField("avg_fraud_amount", DoubleType(), True),
    StructField("computed_ts", TimestampType(), False),
])

print("✅ Silver schema defined -", len(silver_schema.fields), "fields")
print("✅ Gold schema defined -", len(gold_schema.fields), "fields")

✅ Silver schema defined - 18 fields
✅ Gold schema defined - 8 fields


In [8]:
from datetime import datetime

# Test partition strategy - write PaySim data partitioned by date
df_paysim = spark.read.csv(
    "/home/jovyan/work/data/paysim1/PS_20174392719_1491204439457_log.csv",
    header=True,
    inferSchema=True
)

# Add audit columns + partition column
from pyspark.sql.functions import lit, current_timestamp, to_date

df_bronze = df_paysim \
    .withColumn("transaction_id", df_paysim["nameOrig"]) \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source", lit("paysim")) \
    .withColumn("transaction_date", to_date(lit("2024-01-01")))  # PaySim has no real date, use placeholder

# Write to bronze partitioned by date and source
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date", "source") \
    .save("s3a://bronze/transactions/")

print("✅ PaySim data written to bronze layer")
print(f"   Rows: {df_paysim.count():,}")

✅ PaySim data written to bronze layer
   Rows: 6,362,620


In [9]:
# Read back and verify
df_verify = spark.read.format("delta").load("s3a://bronze/transactions/")

print(f"✅ Read back successful")
print(f"   Rows: {df_verify.count():,}")
print(f"   Partitions: {df_verify.rdd.getNumPartitions()}")
print(f"\nSchema:")
df_verify.printSchema()
print(f"\nSample:")
df_verify.show(3)

✅ Read back successful
   Rows: 6,362,620
   Partitions: 6

Schema:
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- transaction_date: date (nullable = true)


Sample:
+----+--------+---------+-----------+-------------+--------------+----------+--------------+--------------+-------+--------------+--------------+--------------------+------+----------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|  nameDest|oldbalanceD